# A2 — Feature & Target Engineering

**Aim:** build a leakage‑free feature table and horizon‑based targets from the frozen 3‑hour master dataset.
This notebook does **not** train or evaluate models.

**Structure:**
1. **Part 1 — Targets:** define binary event targets for multiple horizons using future Kp only.
2. **Part 2 — Features:** create physics‑safe drivers, lags, rolling summaries, and seasonal encoding.
3. **Part 3 — Assembly:** align features and targets, report missingness, and save artifacts.


## Part 1 — Target Construction (Short Explanation)

We build binary targets for each forecast horizon using only future Kp values.
For every time *t*, we look at Kp in the next *H* steps and set `target_H=1`
if the maximum Kp in that future window is at least 5; otherwise `0`.
We then drop the last `max(H)` rows because they lack full future data.
The output is saved as `Data/processed/targets.parquet`, plus a table of event rates.


In [138]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import numpy as np


In [139]:
DATA_DIR = Path("../Data")
INPUT_PATH = DATA_DIR / "processed" / "master_3h.parquet"
OUTPUT_PATH = DATA_DIR / "processed" / "targets.parquet"

START_DATE = "1997-01-01"
HORIZONS = {
    "3h": 1,
    "6h": 2,
    "12h": 4,
    "24h": 8,
    "48h": 16,
    "72h": 24,
    "96h": 32,
}


In [140]:
df = pd.read_parquet(INPUT_PATH)
df = df.sort_values("timestamp_utc").reset_index(drop=True)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
df = df[df["timestamp_utc"] >= START_DATE].reset_index(drop=True)

if "kp" not in df.columns:
    raise KeyError("Expected column 'kp' in master_3h.parquet")

df = df.set_index("timestamp_utc")
kp = df["kp"]


In [141]:
targets = {}

for label, steps in HORIZONS.items():
    future_vals = pd.concat([kp.shift(-i) for i in range(1, steps + 1)], axis=1)
    future_max = future_vals.max(axis=1)
    targets[f"target_{label}"] = (future_max >= 5).astype("int64")

targets_df = pd.DataFrame(targets, index=kp.index)

max_h = max(HORIZONS.values())
targets_df = targets_df.iloc[:-max_h].copy()

rows_dropped = len(kp) - len(targets_df)
print(f"Rows dropped due to horizon truncation: {rows_dropped}")


Rows dropped due to horizon truncation: 32


The “Rows dropped due to horizon truncation: 32” is expected because the
  largest horizon is 96h, which equals 32 steps at a 3‑hour cadence.

In [142]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
targets_df.to_parquet(OUTPUT_PATH)

event_rates = targets_df.mean().to_frame(name="event_rate")
print(event_rates)


            event_rate
target_3h     0.027325
target_6h     0.040178
target_12h    0.060993
target_24h    0.096032
target_48h    0.156304
target_72h    0.211178
target_96h    0.261054


### Note on Class Imbalance

Event rates are low at short horizons (e.g., ~2% at 3h), so targets are class-imbalanced.
This matches the EDA findings in `A1_EDA.ipynb` and is expected for rare storm events.


## Part 2 — Feature Engineering (Physics-Safe)

### 2.1 Raw contemporaneous drivers (t)

Include key drivers at time t without transformation.


In [143]:
RAW_FEATURES = [
    "bz_gsm_3h_min",
    "sw_speed_3h_mean",
    "sw_density_3h_mean",
    "sw_temperature_3h_mean",
    "b_scalar_3h_mean",
    "by_gsm_3h_mean",
]

available_raw = [c for c in RAW_FEATURES if c in df.columns]
missing_raw = [c for c in RAW_FEATURES if c not in df.columns]

features = df[available_raw].copy()

if missing_raw:
    print("Missing optional raw features:", missing_raw)

print("Raw features included:", available_raw)


Raw features included: ['bz_gsm_3h_min', 'sw_speed_3h_mean', 'sw_density_3h_mean', 'sw_temperature_3h_mean', 'b_scalar_3h_mean', 'by_gsm_3h_mean']


### 2.2 Short lags (hard cap = 24h)

Create lagged versions of selected drivers at 3h, 6h, 12h, and 24h.


In [144]:
LAGS = [1, 2, 4, 8]
LAG_VARS = [
    "bz_gsm_3h_min",
    "sw_speed_3h_mean",
    "sw_density_3h_mean",
    "b_scalar_3h_mean",
]

for var in LAG_VARS:
    if var not in df.columns:
        continue
    for lag in LAGS:
        features[f"{var}_lag_{lag}"] = df[var].shift(lag)

print(f"Total features after lags: {features.shape[1]}")


Total features after lags: 22


### 2.3 Rolling summaries (minimal)

Capture short persistence without extending beyond 24h.


**Note (why min vs mean):**
- Bz: strong southward (negative) excursions drive activity, so we keep the *minimum* over the window.
- Speed: sustained elevated flow matters more than spikes, so we use the *mean* over the window.
This matches the qualitative EDA patterns (high Kp with negative Bz and high speed).


In [145]:
if "bz_gsm_3h_min" in df.columns:
    features["bz_min_6h"] = df["bz_gsm_3h_min"].rolling(2).min()
    features["bz_min_12h"] = df["bz_gsm_3h_min"].rolling(4).min()

if "sw_speed_3h_mean" in df.columns:
    features["speed_mean_12h"] = df["sw_speed_3h_mean"].rolling(4).mean()

print(f"Total features after rolling summaries: {features.shape[1]}")


Total features after rolling summaries: 25


### 2.4 Persistence features (allowed)

Include short Kp memory as lagged Kp values (no future leakage).


In [146]:
if "kp" not in df.columns:
    raise KeyError("Expected column 'kp' for persistence features")

features["kp_lag_1"] = df["kp"].shift(1)
features["kp_lag_2"] = df["kp"].shift(2)

print(f"Total features after persistence lags: {features.shape[1]}")


Total features after persistence lags: 27


### 2.5 Seasonal encoding (allowed)

Encode day-of-year with sine/cosine to capture semiannual modulation.


**Note:** The EDA shows a clear seasonal modulation in mean Kp by month;
the sin/cos day-of-year encoding captures this annual cycle smoothly.


In [147]:
doy = features.index.dayofyear
features["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
features["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

print(f"Total features after seasonal encoding: {features.shape[1]}")


Total features after seasonal encoding: 29


## Part 3 — Feature Table Assembly

### Steps
1. Generate all features
2. Align features and targets on timestamp
3. Drop rows with NaNs **after feature creation**
4. Assert no future leakage (sanity checks)


**Note:** We keep NaNs in the feature table here and handle imputation or row
dropping later during modeling (e.g., for sklearn models that require no NaNs).


In [148]:
FEATURES_PATH = DATA_DIR / "processed" / "features.parquet"

# Align on the same timestamp index
features_df = features.copy()
features_df = features_df.loc[targets_df.index]

rows_before = len(features_df)
missing_pct = features_df.isna().mean().sort_values(ascending=False) * 100
rows_with_nan = features_df.isna().any(axis=1).sum()
rows_after_if_drop = rows_before - rows_with_nan

# Keep NaNs here; handle imputation or dropping in modeling
targets_aligned = targets_df.loc[features_df.index]

# Sanity checks: exact index alignment and no extra rows
if not features_df.index.equals(targets_aligned.index):
    raise ValueError("Feature/target index mismatch after alignment")

FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
features_df.to_parquet(FEATURES_PATH)

print(f"Number of features: {features_df.shape[1]}")
print(f"Rows before NaN drop: {rows_before}")
print(f"Rows with any NaN: {rows_with_nan}")
print(f"Rows after NaN drop (if applied): {rows_after_if_drop}")
print("% missing per feature (before drop):")
print(missing_pct)


Number of features: 29
Rows before NaN drop: 84649
Rows with any NaN: 3909
Rows after NaN drop (if applied): 80740
% missing per feature (before drop):
sw_density_3h_mean_lag_8    2.200853
sw_density_3h_mean_lag_4    2.196128
sw_density_3h_mean_lag_2    2.193765
sw_density_3h_mean_lag_1    2.192583
sw_density_3h_mean          2.191402
sw_temperature_3h_mean      1.959858
speed_mean_12h              0.671006
sw_speed_3h_mean_lag_8      0.394571
sw_speed_3h_mean_lag_4      0.389845
sw_speed_3h_mean_lag_2      0.387482
sw_speed_3h_mean_lag_1      0.386301
sw_speed_3h_mean            0.385120
bz_min_12h                  0.373306
bz_min_6h                   0.265803
bz_gsm_3h_min_lag_8         0.218550
bz_gsm_3h_min_lag_4         0.213824
bz_gsm_3h_min_lag_2         0.211461
bz_gsm_3h_min_lag_1         0.210280
bz_gsm_3h_min               0.209099
by_gsm_3h_mean              0.209099
b_scalar_3h_mean_lag_8      0.207917
b_scalar_3h_mean_lag_4      0.203192
b_scalar_3h_mean_lag_2      0.2008

Targets and physics‑safe features are now frozen and aligned by timestamp.
  The feature table contains only information available at time t (plus short
  lags/rolling within 24h and seasonal encoding), and horizon‑specific targets
  are computed using future Kp only. Artifacts Data/processed/targets.parquet
  and Data/processed/features.parquet are saved for downstream modeling.